# 第三章文獻回顧圖表（Plotly）

本 notebook 用於補充「文獻回顧：預測方法」章節之概念圖與整理表。

定位：
- 本 notebook 產生的是文獻回顧用的概念整理圖，不是第五章實驗結果。
- 圖表內容用於說明傳統文本分類、LLM 與 RAG 的方法差異。
- 每張圖表前皆標示建議放置章節與圖說。


In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

COLOR_BLUE = '#4C78A8'
COLOR_RED = '#E45756'
COLOR_GREEN = '#54A24B'
COLOR_ORANGE = '#F58518'
COLOR_TEAL = '#72B7B2'
COLOR_GRAY = '#6B7280'
COLOR_PURPLE = '#7E57C2'

PIC_DIR = Path(r'D:/NTPU_class/paper/pic')
PIC_DIR.mkdir(parents=True, exist_ok=True)


def apply_thesis_theme(fig, height=520, legend_title=None):
    fig.update_layout(
        template='plotly_white',
        height=height,
        font=dict(family='Microsoft JhengHei, Arial', size=14, color='#1F2937'),
        title=dict(font=dict(size=20, color='#111827'), x=0.5, xanchor='center'),
        margin=dict(l=60, r=40, t=90, b=60),
        plot_bgcolor='white',
        paper_bgcolor='white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1, title_text=legend_title),
    )
    fig.update_xaxes(showgrid=True, gridcolor='rgba(0,0,0,0.08)', zeroline=False, linecolor='rgba(0,0,0,0.15)')
    fig.update_yaxes(showgrid=True, gridcolor='rgba(0,0,0,0.08)', zeroline=False, linecolor='rgba(0,0,0,0.15)')
    return fig


def export_fig(fig, filename, width=1200, height=700, scale=2):
    output_path = PIC_DIR / filename
    try:
        fig.write_image(str(output_path), width=width, height=height, scale=scale)
        print(f'exported: {output_path}')
    except Exception as err:
        print(f'圖片輸出失敗：{output_path}')
        print('請確認已安裝 kaleido，例如：pip install -U kaleido')
        print(err)


def make_flow_figure(nodes, arrows, title, height=420):
    fig = go.Figure()
    for node in nodes:
        x, y = node['x'], node['y']
        w, h = node.get('w', 0.17), node.get('h', 0.16)
        fig.add_shape(
            type='rect', xref='paper', yref='paper',
            x0=x - w / 2, x1=x + w / 2, y0=y - h / 2, y1=y + h / 2,
            line=dict(color=node.get('line', '#1F2937'), width=1.5),
            fillcolor=node.get('color', '#F8FAFC'),
            layer='below'
        )
        fig.add_annotation(
            x=x, y=y, xref='paper', yref='paper', showarrow=False,
            text=node['label'], font=dict(size=node.get('size', 14), color='#111827'), align='center'
        )
    for arrow in arrows:
        fig.add_shape(
            type='line', xref='paper', yref='paper',
            x0=arrow['x0'], y0=arrow['y0'], x1=arrow['x1'], y1=arrow['y1'],
            line=dict(color='#374151', width=1.8)
        )
        dx = arrow['x1'] - arrow['x0']
        dy = arrow['y1'] - arrow['y0']
        if abs(dx) >= abs(dy):
            symbol = '▶' if dx >= 0 else '◀'
        else:
            symbol = '▲' if dy >= 0 else '▼'
        fig.add_annotation(
            x=arrow['x1'], y=arrow['y1'], xref='paper', yref='paper',
            text=symbol, showarrow=False, font=dict(size=13, color='#374151')
        )
    fig.update_xaxes(visible=False, range=[0, 1])
    fig.update_yaxes(visible=False, range=[0, 1])
    fig.update_layout(title=title, height=height, margin=dict(l=20, r=20, t=80, b=20), plot_bgcolor='white')
    return fig


## A. 傳統文本分類方法

本段圖表建議放在：**機器學習分類模型** 章節開頭，用於先建立傳統文本分類流程與模型類型概念。


### 圖 3-1 傳統文本分類流程圖

**建議放置章節：機器學習分類模型 開頭**

圖說：本圖整理傳統文本分類方法之基本流程，原始文本通常需先經過前處理與中文斷詞，再轉換為 TF-IDF、Word2Vec 或 fastText 等數值特徵，最後輸入分類模型進行訓練與預測。


In [2]:
nodes = [
    {'label': '原始文本', 'x': 0.08, 'y': 0.55, 'color': '#EEF2FF'},
    {'label': '文本前處理\n清理 / 正規化', 'x': 0.27, 'y': 0.55, 'color': '#F8FAFC'},
    {'label': '中文斷詞\njieba / CKIP', 'x': 0.46, 'y': 0.55, 'color': '#F0FDFA'},
    {'label': '文本特徵表示\nTF-IDF / Word2Vec / fastText', 'x': 0.67, 'y': 0.55, 'w': 0.22, 'color': '#FFF7ED'},
    {'label': '分類模型\nLR / SVM / RF / XGB / MLP', 'x': 0.89, 'y': 0.55, 'w': 0.2, 'color': '#FEF2F2'},
]
arrows = [
    {'x0': 0.17, 'y0': 0.55, 'x1': 0.20, 'y1': 0.55},
    {'x0': 0.36, 'y0': 0.55, 'x1': 0.39, 'y1': 0.55},
    {'x0': 0.55, 'y0': 0.55, 'x1': 0.59, 'y1': 0.55},
    {'x0': 0.78, 'y0': 0.55, 'x1': 0.81, 'y1': 0.55},
]
fig = make_flow_figure(nodes, arrows, '圖 3-1 傳統文本分類流程圖', height=420)
export_fig(fig, 'ch3_traditional_text_classification_flow.png', width=1300, height=420, scale=2)
fig.show()


exported: D:\NTPU_class\paper\pic\ch3_traditional_text_classification_flow.png


### 表 3-1 本研究分類模型特性整理表

**建議放置章節：機器學習分類模型 結尾或各模型介紹前**

表說：本表整理本研究採用之分類模型類型、主要特性與限制，可作為後續各模型小節的總覽。


In [3]:
model_table = pd.DataFrame({
    '模型': ['Logistic Regression', 'SVM-RBF', 'SVM-Poly', 'Random Forest', 'XGBoost', 'MLP'],
    '方法類型': ['線性分類', '最大間隔 / 非線性核', '最大間隔 / 多項式核', '集成學習', '提升式集成學習', '神經網路'],
    '主要優點': ['效率高、可解釋性佳', '適合高維文本特徵', '可捕捉多項式非線性關係', '穩定、可處理非線性', '預測能力佳、具正則化', '可學習複雜非線性關係'],
    '主要限制': ['難捕捉複雜非線性語意', '參數與核函數影響大', '計算成本較高且較難解釋', '高維稀疏文本下成本可能增加', '參數多、調整成本較高', '需較多資料且可解釋性較低'],
    '本研究角色': ['基準模型', '最佳候選模型之一', '非線性比較模型', '集成模型比較', '提升式模型比較', '神經網路比較']
})
display(model_table)

fig = go.Figure(data=[go.Table(
    header=dict(values=list(model_table.columns), fill_color='#1F3A5F', font=dict(color='white', size=13), align='center', height=34),
    cells=dict(values=[model_table[c] for c in model_table.columns], fill_color=[['#F8FAFC', '#FFFFFF']], align='left', height=34, font=dict(size=12, color='#111827'))
)])
fig.update_layout(title='表 3-1 本研究分類模型特性整理表', height=430, margin=dict(l=20, r=20, t=70, b=20))
export_fig(fig, 'ch3_model_characteristics_table.png', width=1500, height=430, scale=2)
fig.show()


,模型,方法類型,主要優點,主要限制,本研究角色
0,Logistic Regression,線性分類,效率高、可解釋性佳,難捕捉複雜非線性語意,基準模型
1,SVM-RBF,最大間隔 / 非線性核,適合高維文本特徵,參數與核函數影響大,最佳候選模型之一
2,SVM-Poly,最大間隔 / 多項式核,可捕捉多項式非線性關係,計算成本較高且較難解釋,非線性比較模型
3,Random Forest,集成學習,穩定、可處理非線性,高維稀疏文本下成本可能增加,集成模型比較
4,XGBoost,提升式集成學習,預測能力佳、具正則化,參數多、調整成本較高,提升式模型比較
5,MLP,神經網路,可學習複雜非線性關係,需較多資料且可解釋性較低,神經網路比較


exported: D:\NTPU_class\paper\pic\ch3_model_characteristics_table.png


### 圖 3-2 分類模型方法類型定位圖

**建議放置章節：機器學習分類模型 結尾**

圖說：本圖以方法類型呈現本研究分類模型之涵蓋範圍，說明模型比較同時納入線性分類、最大間隔方法、集成學習與神經網路等不同技術路徑。


In [4]:
type_df = pd.DataFrame({
    '方法類型': ['線性分類', '最大間隔方法', '集成學習', '神經網路'],
    '模型數量': [1, 2, 2, 1],
    '代表模型': ['Logistic Regression', 'SVM-RBF / SVM-Poly', 'Random Forest / XGBoost', 'MLP']
})
fig = px.bar(
    type_df,
    x='方法類型',
    y='模型數量',
    color='方法類型',
    text='代表模型',
    color_discrete_sequence=[COLOR_BLUE, COLOR_ORANGE, COLOR_GREEN, COLOR_PURPLE],
    title='圖 3-2 分類模型方法類型定位圖',
    labels={'模型數量': '本研究納入模型數量'}
)
fig.update_traces(textposition='outside', cliponaxis=False)
apply_thesis_theme(fig, height=520, legend_title='方法類型')
export_fig(fig, 'ch3_model_type_positioning.png', width=1200, height=520, scale=2)
fig.show()


exported: D:\NTPU_class\paper\pic\ch3_model_type_positioning.png


## B. 文本表示方法

本段圖表可放在傳統文本分類流程說明後，補充 TF-IDF、Word2Vec 與 fastText 的差異。


### 表 3-2 文本表示方法比較表

**建議放置章節：文本表示方法**

表說：本表比較 TF-IDF、Word2Vec 與 fastText 在表示形式、語意能力與限制上的差異，說明本研究為何同時納入多種文本表示方法進行比較。


In [5]:
repr_table = pd.DataFrame({
    '文本表示': ['TF-IDF', 'Word2Vec', 'fastText'],
    '表示概念': ['詞頻與逆文件頻率', '詞向量語意表示', '子詞資訊與詞向量表示'],
    '主要優點': ['簡單、可解釋、適合高維稀疏文本', '可捕捉詞彙語意相近性', '可處理未登錄詞與字詞組成資訊'],
    '主要限制': ['較難捕捉上下文與語意關係', '對未登錄詞較敏感', '仍難完整理解長距離上下文'],
    '本研究用途': ['基準文本表示', '語意向量表示', '強化子詞與語意表示']
})
display(repr_table)

fig = go.Figure(data=[go.Table(
    header=dict(values=list(repr_table.columns), fill_color='#1F3A5F', font=dict(color='white', size=13), align='center', height=34),
    cells=dict(values=[repr_table[c] for c in repr_table.columns], fill_color=[['#F8FAFC', '#FFFFFF']], align='left', height=34, font=dict(size=12, color='#111827'))
)])
fig.update_layout(title='表 3-2 文本表示方法比較表', height=360, margin=dict(l=20, r=20, t=70, b=20))
export_fig(fig, 'ch3_text_representation_comparison_table.png', width=1400, height=360, scale=2)
fig.show()


,文本表示,表示概念,主要優點,主要限制,本研究用途
0,TF-IDF,詞頻與逆文件頻率,簡單、可解釋、適合高維稀疏文本,較難捕捉上下文與語意關係,基準文本表示
1,Word2Vec,詞向量語意表示,可捕捉詞彙語意相近性,對未登錄詞較敏感,語意向量表示
2,fastText,子詞資訊與詞向量表示,可處理未登錄詞與字詞組成資訊,仍難完整理解長距離上下文,強化子詞與語意表示


exported: D:\NTPU_class\paper\pic\ch3_text_representation_comparison_table.png


### 圖 3-3 文本表示方法特性比較圖

**建議放置章節：文本表示方法**

圖說：本圖以概念評分方式比較 TF-IDF、Word2Vec 與 fastText 在可解釋性、語意捕捉、未登錄詞處理與計算效率上的相對特性，用於輔助說明不同文本表示方法的取向差異。


In [6]:
repr_score = pd.DataFrame({
    '文本表示': ['TF-IDF', 'TF-IDF', 'TF-IDF', 'TF-IDF', 'Word2Vec', 'Word2Vec', 'Word2Vec', 'Word2Vec', 'fastText', 'fastText', 'fastText', 'fastText'],
    '特性': ['可解釋性', '語意捕捉', '未登錄詞處理', '計算效率'] * 3,
    '概念分數': [5, 2, 1, 5, 2, 4, 2, 3, 2, 4, 4, 3]
})
fig = px.line_polar(
    repr_score,
    r='概念分數',
    theta='特性',
    color='文本表示',
    line_close=True,
    markers=True,
    color_discrete_sequence=[COLOR_BLUE, COLOR_GREEN, COLOR_ORANGE],
    title='圖 3-3 文本表示方法特性比較圖'
)
fig.update_traces(fill='toself', opacity=0.72)
fig.update_layout(template='plotly_white', height=560, font=dict(family='Microsoft JhengHei, Arial', size=14), title=dict(x=0.5))
export_fig(fig, 'ch3_text_representation_radar.png', width=1100, height=560, scale=2)
fig.show()


exported: D:\NTPU_class\paper\pic\ch3_text_representation_radar.png


## C. LLM 與 RAG

本段圖表建議放在：**大型語言模型與檢索增強生成**。


### 圖 3-4 LLM 與 RAG 在本研究標註流程中的角色圖

**建議放置章節：大型語言模型與檢索增強生成 開頭或 RAG 小節**

圖說：本圖呈現本研究採用 RAG 輔助 LLM 進行情緒勒索文本標註的流程，外部知識庫提供理論依據，LLM 根據原始文本與檢索結果產生標註結果，後續再作為傳統分類模型訓練目標。


In [7]:
nodes = [
    {'label': '原始社群文本', 'x': 0.08, 'y': 0.58, 'color': '#EEF2FF'},
    {'label': '情緒勒索\n知識庫', 'x': 0.28, 'y': 0.78, 'color': '#F0FDFA'},
    {'label': '檢索階段\nRetrieval', 'x': 0.46, 'y': 0.68, 'color': '#ECFDF5'},
    {'label': 'LLM 判斷\nGeneration / Classification', 'x': 0.66, 'y': 0.58, 'w': 0.22, 'color': '#FFF7ED'},
    {'label': '標註結果\n分數 / 類別 / 策略', 'x': 0.88, 'y': 0.58, 'w': 0.2, 'color': '#FEF2F2'},
    {'label': '傳統模型訓練目標', 'x': 0.88, 'y': 0.32, 'w': 0.2, 'color': '#F8FAFC'},
]
arrows = [
    {'x0': 0.17, 'y0': 0.58, 'x1': 0.55, 'y1': 0.58},
    {'x0': 0.34, 'y0': 0.76, 'x1': 0.40, 'y1': 0.70},
    {'x0': 0.52, 'y0': 0.66, 'x1': 0.56, 'y1': 0.60},
    {'x0': 0.77, 'y0': 0.58, 'x1': 0.79, 'y1': 0.58},
    {'x0': 0.88, 'y0': 0.50, 'x1': 0.88, 'y1': 0.40},
]
fig = make_flow_figure(nodes, arrows, '圖 3-4 LLM 與 RAG 在本研究標註流程中的角色圖', height=520)
export_fig(fig, 'ch3_llm_rag_annotation_flow.png', width=1300, height=520, scale=2)
fig.show()


exported: D:\NTPU_class\paper\pic\ch3_llm_rag_annotation_flow.png


### 表 3-3 傳統模型、LLM 與 RAG 比較表

**建議放置章節：大型語言模型與檢索增強生成 結尾**

表說：本表比較傳統文本分類模型、LLM 與 RAG 在輸入需求、知識來源、優點、限制與本研究角色上的差異，用於說明本研究同時使用 RAG 標註與傳統模型預測的理由。


In [8]:
method_table = pd.DataFrame({
    '方法': ['傳統文本分類模型', 'LLM', 'RAG + LLM'],
    '輸入需求': ['斷詞與特徵表示', '原始文本與提示詞', '原始文本、提示詞與檢索知識'],
    '知識來源': ['訓練資料中的統計特徵', '模型預訓練知識與上下文', '外部知識庫與 LLM 語意能力'],
    '主要優點': ['推論速度快、成本低、部署容易', '語意理解能力強、可處理複雜脈絡', '具外部依據、可提升判斷一致性'],
    '主要限制': ['依賴特徵工程與標註資料', '成本較高且可能產生幻覺', '流程較複雜且受知識庫品質影響'],
    '本研究角色': ['最終分類模型與速度比較對象', '輔助語意判斷', '建立較具理論依據的標註流程']
})
display(method_table)

fig = go.Figure(data=[go.Table(
    header=dict(values=list(method_table.columns), fill_color='#1F3A5F', font=dict(color='white', size=13), align='center', height=34),
    cells=dict(values=[method_table[c] for c in method_table.columns], fill_color=[['#F8FAFC', '#FFFFFF']], align='left', height=38, font=dict(size=12, color='#111827'))
)])
fig.update_layout(title='表 3-3 傳統模型、LLM 與 RAG 比較表', height=430, margin=dict(l=20, r=20, t=70, b=20))
export_fig(fig, 'ch3_traditional_llm_rag_comparison_table.png', width=1500, height=430, scale=2)
fig.show()


,方法,輸入需求,知識來源,主要優點,主要限制,本研究角色
0,傳統文本分類模型,斷詞與特徵表示,訓練資料中的統計特徵,推論速度快、成本低、部署容易,依賴特徵工程與標註資料,最終分類模型與速度比較對象
1,LLM,原始文本與提示詞,模型預訓練知識與上下文,語意理解能力強、可處理複雜脈絡,成本較高且可能產生幻覺,輔助語意判斷
2,RAG + LLM,原始文本、提示詞與檢索知識,外部知識庫與 LLM 語意能力,具外部依據、可提升判斷一致性,流程較複雜且受知識庫品質影響,建立較具理論依據的標註流程


exported: D:\NTPU_class\paper\pic\ch3_traditional_llm_rag_comparison_table.png


### 圖 3-5 傳統模型、LLM 與 RAG 方法特性比較圖

**建議放置章節：大型語言模型與檢索增強生成 結尾**

圖說：本圖以概念評分方式呈現傳統模型、LLM 與 RAG 在推論速度、語意理解、可追溯性與系統複雜度上的相對特性。此圖可用於銜接本研究後續設計：先以 RAG 輔助標註，再訓練傳統模型以降低推論成本。


In [9]:
method_score = pd.DataFrame({
    '方法': ['傳統模型'] * 4 + ['LLM'] * 4 + ['RAG + LLM'] * 4,
    '特性': ['推論速度', '語意理解', '可追溯性', '系統複雜度'] * 3,
    '概念分數': [5, 2, 2, 2, 2, 5, 2, 3, 1, 5, 5, 5]
})
fig = px.line_polar(
    method_score,
    r='概念分數',
    theta='特性',
    color='方法',
    line_close=True,
    markers=True,
    color_discrete_sequence=[COLOR_BLUE, COLOR_RED, COLOR_GREEN],
    title='圖 3-5 傳統模型、LLM 與 RAG 方法特性比較圖'
)
fig.update_traces(fill='toself', opacity=0.72)
fig.update_layout(template='plotly_white', height=560, font=dict(family='Microsoft JhengHei, Arial', size=14), title=dict(x=0.5))
export_fig(fig, 'ch3_traditional_llm_rag_radar.png', width=1100, height=560, scale=2)
fig.show()


exported: D:\NTPU_class\paper\pic\ch3_traditional_llm_rag_radar.png


## 備註

- 本 notebook 圖表為文獻回顧與研究設計說明用途，屬於概念整理，不代表第五章實驗結果。
- 若論文圖號不是第三章，可將 `圖 3-x` 與 `表 3-x` 依實際章節統一調整。
- 圖 3-3 與圖 3-5 的概念分數是文獻特性整理，不應寫成實證量化結果。
